### Main pack setup and pack dependencies
# using  AstroTLPlot

In [ ]:
cd("/home/tomaslima/julia_tlima/pkg/AstroTLPlot/")
import Pkg;
using Pkg;
Pkg.activate(".")
using Revise  # Opcional (veja abaixo)
using AstroTLPlot

using Dates
using FileIO
using Printf

Pkg.add("Colors")

using Colors
using LinearAlgebra
using Statistics
using DelimitedFiles

using Makie
using CairoMakie # Often needed for display in certain environments like VS Code or Pluto
using LaTeXStrings # For proper LaTeX rendering of scientific symbols

### Read config file and data files
## Setup the strcuts

In [ ]:
# 1- call namelists
        # Carregar e testar a função
        file_name = "./data/config/indatpgp.yaml" 
        
       
# Call the function and store the result (Tuple on success, Int on error)
result = load_simulation_config(file_name)
all_config=load_simulation_config_struct(file_name)


    # 1. Check if the result is an error code (Int)
    if isa(result, Int)
    
    # Determine the appropriate error code for display.
    error_code_to_display = result
    
    # 2. If the returned code is not mapped in the dictionary, use the generic ERROR_UNKNOWN (Code 1)
    if !haskey(AstroTLPlot.ERROR_MESSAGES_DICT, result)
        error_code_to_display = AstroTLPlot.ERROR_UNKNOWN 
    end
 
    # Print the error message using the determined code
    println("ERROR loading configuration: ", AstroTLPlot.ERROR_MESSAGES_DICT[error_code_to_display])
    
    # Optional: Exit the program or load default settings here
    # exit(1) 
    
    else
        # 3. If it's not an error code, it's the data tuple (Success)
        # Safely destructure the data
        config_data, pgp_data, runtime_data, modions_data = result
    
        #println("Configurations loaded successfully.")
        println(AstroTLPlot.ERROR_MESSAGES_DICT[AstroTLPlot.STATUS_SUCCESS])
        # Example usage:
        println("Directory: ", config_data.directories.directory) 
    end


# 2- call configure
        # Calcular
        _increments, _set_min_max_index, vol_local, vol_global, tr = configure(config_data, pgp_data, runtime_data)

        #-----

 # 3 - call allocate_vars 
        # allocate
        simulations_data, time_files = allocate_vars(config_data, pgp_data, runtime_data)

        # read ref data
        # Nome do ficheiro e número de entradas que queres ler
        filename="/home/tomaslima/julia_tlima/pkg/AstroTLPlot/data/input/ref/ref.dat"

 # 4- call readlist  
        timescale = config_data.scales.timescale
        nfiles =  config_data.number_of_plots.nfiles

        # Ler os dados de ref.dat   generate TimeFile ==> time::Vector{Float64} , files::Vector{Int} 
        timefile = read_ref_data_ds(filename, nfiles, timescale)

# 5. Call the data reading function (readdata_hdf5!)

    # Call the function and store the result. It will be 'nothing' on success, or an 'Int' error code on failure.
    read_result = readdata_hdf5!(simulations_data, config_data, runtime_data)
    
    # Check if the result is an error code (Int)
    if isa(read_result, Int)
        
        # Determine the appropriate error code for display.
        error_code_to_display = read_result
        
        # Check if the returned code is mapped; if not, use the generic ERROR_UNKNOWN (Code 1).
        if !haskey(AstroTLPlot.ERROR_MESSAGES_DICT, read_result)
            error_code_to_display = ERROR_UNKNOWN 
        end
    
        # Print the error message using the determined code
        println("CRITICAL ERROR during HDF5 data reading: ", AstroTLPlot.ERROR_MESSAGES_DICT[error_code_to_display])
        
        # Here, you would typically halt execution due to a failure to load simulation data.
        # exit(1) 
        
    else
        # The result is 'nothing' (or any non-integer value), indicating success.
        println(AstroTLPlot.ERROR_MESSAGES_DICT[AstroTLPlot.STATUS_SUCCESS])
        println("All simulation data is ready for processing.")
    end

 # Variables  readdata.jl.... tem, pre , vel2
        variables_v2!( simulations_data,config_data)


### Ions and Electrons

In [ ]:
 # 6 - Start the ionization tables

        #  elem=Element(n_flags=26)
        #  count_ions!(config_data,elem)
        elem = count_ions(config_data)

       # tps = ions_read_vf(config_data,pgp_data,runtime_data,modions_data,elem)
 
        #Create an instance using the outer constructor
        # tp = TemperatureProperties(in_size, jn_size, kn_size, nelem, kernmax;lele=lele_flag)

         #tps = ions_read(config_data,pgp_data,runtime_data,modions_data,elem)
         abundances!(config_data,elem)    
        
         #temp_props = TemperatureProperties(in_size, jn_size, kn_size, elem.nelem, elem.kernmax; 
         #        lele=cfg_data.variable_params.lele)

        # 7 - call loopfiles
      
         tps = ions_read_vf(config_data,pgp_data,runtime_data,modions_data,elem)

        #Create an instance using the outer constructor
           #  tp = TemperatureProperties(in_size, jn_size, kn_size, nelem, kernmax; lele=lele_flag)

         #tps = ions_read(config_data,pgp_data,runtime_data,modions_data,elem)
        
        ionp=create_ionproperties()
        


### Plot elemnt heatmap + contour + combine ( heat + cont) PDF


# Simulation Batch Export — Quick Guide (Notebook-Friendly)

This notebook call runs a batch export of figures for a list of variables and saves the resulting images to a structured folder tree. It is designed for quick, repeatable generation of plots in one pass.

---

## What the call does

```julia
process_simulation_list( 
    simulations_data;
    kern = 0,
    ionz = 0,
    is_ions = false,
    cfg = config_data,
    pgp = pgp_data,
    rt = runtime_data,
    modions = modions_data,  
    variable_names = vars,
    formats = ["png"],
    save_path = "./output/maps"
)


# Pre-validated list (example)
vars = ["den", "pre", "tem"]

# Extended list (ensure names are valid in your stack)
vars = ["den", "mach", "pok", "pre", "ram", "rok", "tem"]

### Folder structure per variable
└── maps
    ├── den
    │   ├── color
    │   │   └── den_003000-042_221225_1405.png
    │   ├── contour
    │   │   └── den_003000-042_221225_1405.png
    │   ├── heat_cont
    │   │   └── den_003000-042_221225_1405.png
    │   ├── pdf
    │   │   └── den_003000-042_221225_1405.png
    │   └── statistics
    │       └── den_003000-042_221225_1405.txt
    ├── pre
    │   ├── color
    │   │   └── pre_003000-042_221225_1405.png
    │   ├── contour
    │   │   └── pre_003000-042_221225_1405.png
    │   ├── heat_cont
    │   │   └── pre_003000-042_221225_1405.png
    │   ├── pdf
    │   │   └── pre_003000-042_221225_1405.png
    │   └── statistics
    │       └── pre_003000-042_221225_1405.txt
    └── tem
        ├── color
        │   └── tem_003000-042_221225_1406.png
        ├── contour
        │   └── tem_003000-042_221225_1406.png
        ├── heat_cont
        │   └── tem_003000-042_221225_1406.png
        ├── pdf
        │   └── tem_003000-042_221225_1406.png
        └── statistics
            └── tem_003000-042_221225_1406.txt
  ...

In [ ]:

#vars=["den", "mach","pok","pre", "ram","rok","tem"]
vars = ["den", "pre", "tem"]
process_simulation_list(
    simulations_data;
    kern = 0,
    ionz = 0,
    is_ions = false,
    cfg = config_data,
    pgp = pgp_data,
    rt = runtime_data,
    modions = modions_data,
    variable_names = vars,
    formats = ["png"],
    save_path = "./data/output/maps"
)


In [ ]:

# Lista de formatos (opcional)
formats = ["png"]  # pode usar ["pdf", "png"] se quiser ambos

# Pasta base onde o subplot será guardado (a função criará ./data/output/maps/subplot/den)
save_base = "./output/maps"

# Chamada para gerar os subplots da variável 'den' (TOP view)
maps_subplot!(
    simulations_data.den,            # 3D array: data
    simulations_data.X_grid,         # xgrid
    simulations_data.Y_grid,         # ygrid
    simulations_data.Z_grid,         # zgrid
    "den";                           # var_name
    cfg = config_data,               # ConfigData
    pgp = pgp_data,                  # PGPData
    rt  = runtime_data,              # RuntimeData
    modions = modions_data,          # ModionsData
    formats = formats,               # ["    formats = formats,               # ["png"] por defeito
    base_save_path = save_base       # ./data/output/maps/subplot
)

In [ ]:

# mapas_subplot_news_v3!(simulations_data.den,simulations_data.X_grid,simulations_data.Y_grid,simulations_data.Z_grid,"den";
# cfg=config_data,pgp=pgp_data,rt=runtime_data,modions=modions_data)
# Lista de formatos (opcional)
formats = ["png"]  # pode usar ["pdf", "png"] se quiser ambos

# Pasta base onde o subplot será guardado (a função criará ./data/output/maps/subplot/den)
save_base = "./output"

# Chamada para gerar os subplots da variável 'den' (TOP view)
maps_subplot!(
    simulations_data.ene,            # 3D array: data
    simulations_data.X_grid,         # xgrid
    simulations_data.Y_grid,         # ygrid
    simulations_data.Z_grid,         # zgrid
    "pre";                           # var_name
    cfg = config_data,               # ConfigData
    pgp = pgp_data,                  # PGPData
    rt  = runtime_data,              # RuntimeData
    modions = modions_data,          # ModionsData
    formats = formats,               # ["    formats = formats,               # ["png"] por defeito
    base_save_path = save_base            # ./data/output/maps/subplot
)


### Ionization Species Analysis: ions!
The ions! function is the core routine for calculating and visualizing the spatial distribution of specific ion species within the simulation grid. It processes the ionization fractions, applies physical scaling, and generates multi-view maps (Top, Front, Side).
## Function Signature
ions!(ionp, tps, elem, sml, cfg, pgp, rt, modions; 
      formats=["png"], 
      base_save_path="./data/output/ions")

├── C
│   ├── C00+_003000-042_221225_1421.png
│   ├── C01+_003000-042_221225_1422.png
│   ├── C02+_003000-042_221225_1422.png
│   ├── C03+_003000-042_221225_1422.png
│   ├── C04+_003000-042_221225_1423.png
│   ├── C05+_003000-042_221225_1423.png
│   └── C06+_003000-042_221225_1423.png
├── den_003000-042_221225_1420.png
├── den_003000-042_221225_1421.png
├── den_003000-042_221225_1422.png
├── den_003000-042_221225_1423.png
├── den_003000-042_221225_1424.png
├── den_003000-042_221225_1425.png
├── den_003000-042_221225_1426.png
├── H
│   ├── H00+_003000-042_221225_1420.png
│   └── H01+_003000-042_221225_1420.png
├── He
│   ├── He00+_003000-042_221225_1420.png
│   ├── He01+_003000-042_221225_1421.png
│   └── He02+_003000-042_221225_1421.png
├── N
│   ├── N00+_003000-042_221225_1424.png
│   ├── N01+_003000-042_221225_1424.png
│   ├── N02+_003000-042_221225_1424.png
│   ├── N03+_003000-042_221225_1424.png
│   ├── N04+_003000-042_221225_1425.png
│   ├── N05+_003000-042_221225_1425.png
│   ├── N06+_003000-042_221225_1425.png
│   └── N07+_003000-042_221225_1426.png
└── statistics
    ├── C00+_003000-042_221225_1421.txt
    ├── C01+_003000-042_221225_1422.txt
    ├── C02+_003000-042_221225_1422.txt
    ├── C03+_003000-042_221225_1422.txt
    ├── C04+_003000-042_221225_1423.txt
    ├── C05+_003000-042_221225_1423.txt
    ├── C06+_003000-042_221225_1423.txt
    ├── H00+_003000-042_221225_1420.txt
    ├── H01+_003000-042_221225_1420.txt
    ├── He00+_003000-042_221225_1420.txt
    ├── He01+_003000-042_221225_1421.txt
    ├── He02+_003000-042_221225_1421.txt
    ├── N00+_003000-042_221225_1424.txt
    ├── N01+_003000-042_221225_1424.txt
    ├── N02+_003000-042_221225_1424.txt
    ├── N03+_003000-042_221225_1424.txt
    ├── N04+_003000-042_221225_1425.txt
    ├── N05+_003000-042_221225_1425.txt
    ├── N06+_003000-042_221225_1425.txt
    └── N07+_003000-042_221225_1426.txt



In [ ]:
ions!(ionp,tps,elem,simulations_data,config_data,pgp_data,runtime_data,modions_data)

### Ionization State Visualization: plot_element_subplot
This function creates a comprehensive grid of subplots, where each cell represents a specific ionization state of a chosen element.

In [ ]:

plot_element_subplot(
    tps,                  # TemperatureProperties
    simulations_data,     # SimulationData
    elem,                 # Element
    7;                    # ee_num (Índice do estado de ionização)
    # Parâmetros Opcionais (Keyword Arguments)
    ncols = 4,
    save_dir = "./data/output/maps/subplots",
    formats = ["png"]           # Exporta em múltiplos formatos
)

### Physical Modeling: Electron Density and Ionization Mapping
This notebook demonstrates the usage of two core functions for plasma simulation analysis: electron! for density computation and plot_element_subplot for advanced visualization of ionization states.
produce the output:
├── ele
│   ├── color
│   │   ├── ele_003000-042_221225_1605.png
│   │   └── ele_003000-042_221225_1612.png
│   ├── contour
│   │   ├── ele_003000-042_221225_1605.png
│   │   └── ele_003000-042_221225_1612.png
│   ├── heat_cont
│   │   ├── ele_003000-042_221225_1605.png
│   │   └── ele_003000-042_221225_1612.png
│   ├── pdf
│   │   ├── ele_003000-042_221225_1605.png
│   │   └── ele_003000-042_221225_1612.png
│   └── statistics
│       ├── ele_003000-042_221225_1605.txt
│       └── ele_003000-042_221225_1612.txt
└── elez
    ├── color
    │   ├── elez=08-003000-042_221225_1612.png
    │   ├── elez=08-003000-042_221225_1613.png
    │   └── elez=08-003000-042_221225_1614.png
    ├── contour
    │   ├── elez=08-003000-042_221225_1612.png
    │   ├── elez=08-003000-042_221225_1613.png
    │   └── elez=08-003000-042_221225_1614.png
    ├── heat_cont
    │   ├── elez=08-003000-042_221225_1612.png
    │   ├── elez=08-003000-042_221225_1613.png
    │   └── elez=08-003000-042_221225_1614.png
    ├── pdf
    │   ├── elez=01-003000-042_221225_1612.png
    │   ├── elez=01-003000-042_221225_1613.png
    │   └── elez=01-003000-042_221225_1614.png
    └── statistics
        ├── elez=08-003000-042_221225_1612.txt
        ├── elez=08-003000-042_221225_1613.txt
        └── elez=08-003000-042_221225_1614.txt




In [ ]:
electron!(
    tps,              # TemperatureProperties (será atualizado in-place)
    elem,             # Element
    simulations_data, # SimulationData
    config_data,      # ConfigData
    pgp_data,         # PGPData
    runtime_data,     # RuntimeData (controla se gera plots via rt.output_plot)
    modions_data;     # ModionsData

    # Parâmetros Opcionais (Keyword Arguments)
    formats = ["png"],                # NEW: Formatos de saída formats = ["png","pdf","svg"]
    save_path = "./data/output/electrons" # NEW: Pasta base para os outputs
    )